# Fine-tune YOLOv8s-World on Objects365

Fine-tunes **YOLOv8s-WorldV2** on the [Objects365](https://universe.roboflow.com/objects-365-consortium/objects-365) dataset (365 object categories) sourced from **Roboflow Universe**.
YOLOWorld uses CLIP text embeddings — fine-tuning here makes `set_classes()` much more accurate on everyday objects beyond the COCO-80 baseline.

The Roboflow download delivers data **pre-formatted in YOLOv8 format** with a ready-made `data.yaml`, so no COCO → YOLO conversion step is needed.

---
### Quick-start
1. **Runtime → Change runtime type → T4 GPU** (free tier is sufficient for a subset run)
2. Paste your **Roboflow API key** in Section 3 (free account at [roboflow.com](https://roboflow.com))
3. Run all cells top to bottom
4. Download the trained `.pt` / `.onnx` from the last cell

> **Re-runnable**: if the download is interrupted, just re-run Section 5 — `wget -c` resumes from where it left off and the extraction step is skipped if already complete.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0 — Environment Check

In [8]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('Running in Colab:', IN_COLAB)

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip() or 'not found')
except FileNotFoundError:
    print('GPU: nvidia-smi not available (CPU only)')

print('Python:', sys.version.split()[0])

Running in Colab: True
GPU: nvidia-smi not available (CPU only)
Python: 3.12.13


## 1 — Install Dependencies

In [ ]:
%pip install -q ultralytics>=8.2 requests pyyaml tqdm pillow

## 2 — (Optional) Mount Google Drive for Persistence

Recommended so checkpoints and the downloaded dataset survive session resets.

In [ ]:
import os

USE_DRIVE = False   # ← set True to save everything to Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/yoloworld_obj365'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive mounted at:', DRIVE_ROOT)
else:
    DRIVE_ROOT = None
    print('Drive not mounted — outputs will stay in /content (lost on disconnect)')

## 3 — Configuration

In [ ]:
import os, torch
from pathlib import Path

# ─── Roboflow credentials ─────────────────────────────────────────────────────
# Get a free API key at https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'   # ← paste your key
RF_WORKSPACE     = 'objects-365-consortium'
RF_PROJECT       = 'objects-365'
DATASET_VERSION  = 1     # check available versions on Roboflow Universe

# ─── Root directory ───────────────────────────────────────────────────────────
ROOT = Path(DRIVE_ROOT) if USE_DRIVE and DRIVE_ROOT else Path('/content/obj365_finetune')
ROOT.mkdir(parents=True, exist_ok=True)

DATASET_DIR = ROOT / 'dataset'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = ROOT / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Model & training ─────────────────────────────────────────────────────────
BASE_MODEL  = 'yolov8s-worldv2.pt'
EPOCHS      = 50
IMG_SIZE    = 640
BATCH       = 16        # reduce to 8 or 4 if OOM on T4
WORKERS     = 2
EXP_NAME    = 'obj365_world'

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Root   : {ROOT}')

## 4 — Official Objects365 Class Map (365 classes)

Reference class mapping from the Ultralytics `Objects365.yaml`.

In [ ]:
OBJECTS365_CLASSES = [
    'Person', 'Sneakers', 'Chair', 'Other Shoes', 'Hat', 'Car', 'Lamp',
    'Glasses', 'Bottle', 'Desk', 'Cup', 'Street Lights', 'Cabinet/shelf',
    'Handbag/Satchel', 'Bracelet', 'Plate', 'Picture/Frame', 'Helmet',
    'Book', 'Gloves', 'Storage box', 'Boat', 'Leather Shoes', 'Flower',
    'Bench', 'Potted Plant', 'Bowl/Basin', 'Flag', 'Pillow', 'Boots',
    'Vase', 'Microphone', 'Necklace', 'Ring', 'SUV', 'Wine Glass', 'Belt',
    'Monitor/TV', 'Backpack', 'Umbrella', 'Traffic Light', 'Speaker',
    'Watch', 'Tie', 'Trash bin Can', 'Slippers', 'Bicycle', 'Stool',
    'Barrel/bucket', 'Van', 'Couch', 'Sandals', 'Basket', 'Drum',
    'Pen/Pencil', 'Bus', 'Wild Bird', 'High Heels', 'Motorcycle', 'Guitar',
    'Carpet', 'Cell Phone', 'Bread', 'Camera', 'Canned', 'Truck',
    'Traffic cone', 'Cymbal', 'Lifesaver', 'Towel', 'Stuffed Toy', 'Candle',
    'Sailboat', 'Laptop', 'Awning', 'Bed', 'Faucet', 'Tent', 'Horse',
    'Mirror', 'Power outlet', 'Sink', 'Apple', 'Air Conditioner', 'Knife',
    'Hockey Stick', 'Paddle', 'Pickup Truck', 'Fork', 'Traffic Sign',
    'Balloon', 'Tripod', 'Dog', 'Spoon', 'Clock', 'Pot', 'Cow', 'Cake',
    'Dining Table', 'Sheep', 'Hanger', 'Blackboard/Whiteboard', 'Napkin',
    'Other Fish', 'Orange/Tangerine', 'Toiletry', 'Keyboard', 'Tomato',
    'Lantern', 'Machinery Vehicle', 'Fan', 'Green Vegetables', 'Banana',
    'Baseball Glove', 'Airplane', 'Mouse', 'Train', 'Pumpkin', 'Soccer',
    'Skiboard', 'Luggage', 'Nightstand', 'Tea pot', 'Telephone', 'Trolley',
    'Head Phone', 'Sports Car', 'Stop Sign', 'Dessert', 'Scooter',
    'Stroller', 'Crane', 'Remote', 'Refrigerator', 'Oven', 'Lemon', 'Duck',
    'Baseball Bat', 'Surveillance Camera', 'Cat', 'Jug', 'Broccoli',
    'Piano', 'Pizza', 'Elephant', 'Skateboard', 'Surfboard', 'Gun',
    'Skating and Skiing shoes', 'Gas stove', 'Donut', 'Bow Tie', 'Carrot',
    'Toilet', 'Kite', 'Strawberry', 'Other Balls', 'Shovel', 'Pepper',
    'Computer Box', 'Toilet Paper', 'Cleaning Products', 'Chopsticks',
    'Microwave', 'Pigeon', 'Baseball', 'Cutting/chopping Board',
    'Coffee Table', 'Side Table', 'Scissors', 'Marker', 'Pie', 'Ladder',
    'Snowboard', 'Cookies', 'Radiator', 'Fire Hydrant', 'Basketball',
    'Zebra', 'Grape', 'Giraffe', 'Potato', 'Sausage', 'Tricycle', 'Violin',
    'Egg', 'Fire Extinguisher', 'Candy', 'Fire Truck', 'Billiards',
    'Converter', 'Bathtub', 'Wheelchair', 'Golf Club', 'Briefcase',
    'Cucumber', 'Cigar/Cigarette', 'Paint Brush', 'Pear', 'Heavy Truck',
    'Hamburger', 'Extractor', 'Extension Cord', 'Tong', 'Tennis Racket',
    'Folder', 'American Football', 'earphone', 'Mask', 'Kettle', 'Tennis',
    'Ship', 'Swing', 'Coffee Machine', 'Slide', 'Carriage', 'Onion',
    'Green beans', 'Projector', 'Frisbee',
    'Washing Machine/Drying Machine', 'Chicken', 'Printer', 'Watermelon',
    'Saxophone', 'Tissue', 'Toothbrush', 'Ice cream', 'Hot-air balloon',
    'Cello', 'French Fries', 'Scale', 'Trophy', 'Cabbage', 'Hot dog',
    'Blender', 'Peach', 'Rice', 'Wallet/Purse', 'Volleyball', 'Deer',
    'Goose', 'Tape', 'Tablet', 'Cosmetics', 'Trumpet', 'Pineapple',
    'Golf Ball', 'Ambulance', 'Parking meter', 'Mango', 'Key', 'Hurdle',
    'Fishing Rod', 'Medal', 'Flute', 'Brush', 'Penguin', 'Megaphone',
    'Corn', 'Lettuce', 'Garlic', 'Swan', 'Helicopter', 'Green Onion',
    'Sandwich', 'Nuts', 'Speed Limit Sign', 'Induction Cooker', 'Broom',
    'Trombone', 'Plum', 'Rickshaw', 'Goldfish', 'Kiwi fruit',
    'Router/modem', 'Poker Card', 'Toaster', 'Shrimp', 'Sushi', 'Cheese',
    'Notepaper', 'Cherry', 'Pliers', 'CD', 'Pasta', 'Hammer', 'Cue',
    'Avocado', 'Hami melon', 'Flask', 'Mushroom', 'Screwdriver', 'Soap',
    'Recorder', 'Bear', 'Eggplant', 'Board Eraser', 'Coconut',
    'Tape Measure/Ruler', 'Pig', 'Showerhead', 'Globe', 'Chips', 'Steak',
    'Crosswalk Sign', 'Stapler', 'Camel', 'Formula 1', 'Pomegranate',
    'Dishwasher', 'Crab', 'Hoverboard', 'Meatball', 'Rice Cooker', 'Tuba',
    'Calculator', 'Papaya', 'Antelope', 'Parrot', 'Seal', 'Butterfly',
    'Dumbbell', 'Donkey', 'Lion', 'Urinal', 'Dolphin', 'Electric Drill',
    'Hair Dryer', 'Egg tart', 'Jellyfish', 'Treadmill', 'Lighter',
    'Grapefruit', 'Game board', 'Mop', 'Radish', 'Baozi', 'Target',
    'French', 'Spring Rolls', 'Monkey', 'Rabbit', 'Pencil Case', 'Yak',
    'Red Cabbage', 'Binoculars', 'Asparagus', 'Barbell', 'Scallop',
    'Noddles', 'Comb', 'Dumpling', 'Oyster', 'Table Tennis paddle',
    'Cosmetics Brush/Eyeliner Pencil', 'Chainsaw', 'Eraser', 'Lobster',
    'Durian', 'Okra', 'Lipstick', 'Cosmetics Mirror', 'Curling',
    'Table Tennis',
]

assert len(OBJECTS365_CLASSES) == 365, f'Expected 365, got {len(OBJECTS365_CLASSES)}'
print(f'Reference class list: {len(OBJECTS365_CLASSES)} classes')
print('First 5:', OBJECTS365_CLASSES[:5])
print('Last  5:', OBJECTS365_CLASSES[-5:])

## 5 — Download Objects365 from Roboflow (resumable)

**Why not `roboflow.version.download()`?**  
The Roboflow SDK wraps a plain HTTP download with no resume support — if Colab's connection drops mid-way it restarts from 0%.

Instead we:
1. Call the Roboflow REST API to get a **pre-signed download URL**
2. Use `wget -c` (continue / resume) to download the zip
3. Extract only if not already done (idempotent — safe to re-run)

Dataset page: https://universe.roboflow.com/objects-365-consortium/objects-365

In [ ]:
import requests, zipfile, json, time
from pathlib import Path

ZIP_PATH    = DATASET_DIR / 'roboflow_obj365.zip'
EXTRACT_DIR = DATASET_DIR / 'extracted'
SENTINEL    = EXTRACT_DIR / '.done'   # written after successful extraction

# ── Step 1: poll Roboflow API until the export zip is ready ──────────────────
# Roboflow builds the zip on-demand. While it is generating it returns:
#   {"progress": 0.0 … 1.0}
# Once the zip is ready it returns:
#   {"export": {"link": "https://..."}}
# We poll until we see the link (or give up after MAX_WAIT seconds).

api_url = (
    f'https://api.roboflow.com/{RF_WORKSPACE}/{RF_PROJECT}'
    f'/{DATASET_VERSION}/yolov8'
)

POLL_INTERVAL = 10     # seconds between retries
MAX_WAIT      = 1800   # give up after 30 minutes

print('Waiting for Roboflow to prepare the export (can take a few minutes on first request)...')
t_start      = time.time()
download_url = None

while (time.time() - t_start) < MAX_WAIT:
    resp = requests.get(api_url, params={'api_key': ROBOFLOW_API_KEY}, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    if 'export' in payload and 'link' in payload.get('export', {}):
        download_url = payload['export']['link']
        break

    progress = payload.get('progress', '?')
    elapsed  = int(time.time() - t_start)
    print(f'  Export progress: {float(progress) * 100:.0f}%  ({elapsed}s elapsed)'
          f' — retrying in {POLL_INTERVAL}s...', end='\r')
    time.sleep(POLL_INTERVAL)

print()  # newline after \r progress line

if download_url is None:
    print('Last API response:', json.dumps(payload, indent=2))
    raise RuntimeError(
        f'Roboflow export did not finish within {MAX_WAIT}s.\n'
        'Try increasing MAX_WAIT or re-run this cell later.'
    )

print(f'Export ready.')
print(f'URL: {download_url[:80]}...')

# ── Step 2: download with wget -c (resume-capable) ────────────────────────────
if SENTINEL.exists():
    print('\nDataset already extracted — skipping download & extraction.')
else:
    print(f'\nDownloading to {ZIP_PATH}')
    print('(wget -c resumes automatically if the connection drops — just re-run this cell)')
    !wget -c --show-progress -O "{ZIP_PATH}" "{download_url}"

    # Verify zip integrity before extracting
    print('Verifying zip integrity...')
    try:
        with zipfile.ZipFile(ZIP_PATH) as zf:
            bad = zf.testzip()
        if bad:
            raise RuntimeError(
                f'Corrupt entry in zip: {bad}\n'
                f'Delete {ZIP_PATH} and re-run.'
            )
    except zipfile.BadZipFile:
        raise RuntimeError(
            f'{ZIP_PATH} is not a valid zip — download may be incomplete.\n'
            f'Delete the file and re-run this cell to retry:\n'
            f'  !rm "{ZIP_PATH}"'
        )

    # ── Step 3: extract ───────────────────────────────────────────────────────
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR)
    SENTINEL.touch()
    print('Extraction complete.')

# ── Step 4: locate data.yaml ──────────────────────────────────────────────────
yaml_files = sorted(EXTRACT_DIR.rglob('data.yaml'))
if not yaml_files:
    raise FileNotFoundError(
        f'No data.yaml found under {EXTRACT_DIR}.\n'
        'The zip may not have extracted correctly — delete the .done sentinel and retry:\n'
        f'  !rm "{SENTINEL}"'
    )

DATA_YAML_PATH = yaml_files[0]
DATASET_ROOT   = DATA_YAML_PATH.parent

print(f'\nDataset root : {DATASET_ROOT}')
print(f'data.yaml    : {DATA_YAML_PATH}')

## 6 — Inspect and Patch data.yaml

Roboflow may shorten class names or use a subset.  
Set `PATCH_NAMES = True` to replace them with the official 365-class list.

In [ ]:
import yaml

with open(DATA_YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)

rf_names    = data_cfg.get('names', [])
NUM_CLASSES = data_cfg.get('nc', len(rf_names))

print(f'Roboflow nc    : {NUM_CLASSES}')
print(f'Roboflow names : {rf_names[:5]} ... {rf_names[-5:]}')
print(f'Reference nc   : 365')

# Set True only if Roboflow names differ from official AND nc == 365
PATCH_NAMES = False

if PATCH_NAMES and NUM_CLASSES == 365:
    data_cfg['names'] = OBJECTS365_CLASSES
    with open(DATA_YAML_PATH, 'w') as f:
        yaml.dump(data_cfg, f, default_flow_style=False, allow_unicode=True)
    print('data.yaml patched with official 365-class names.')

CLASSES = data_cfg['names']
print(f'\nUsing {len(CLASSES)} classes for training.')

## 7 — Inspect Label Distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
from pathlib import Path

# Roboflow uses 'valid'; fall back to 'val'
LBL_VAL = DATASET_ROOT / 'valid' / 'labels'
if not LBL_VAL.exists():
    LBL_VAL = DATASET_ROOT / 'val' / 'labels'

def count_labels(labels_dir: Path) -> Counter:
    c = Counter()
    for f in labels_dir.glob('*.txt'):
        for line in f.read_text().splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    return c

val_counts  = count_labels(LBL_VAL)
total_boxes = sum(val_counts.values())
print(f'Val split: {len(val_counts)} distinct classes | {total_boxes:,} total boxes')

top_n = 30
top = val_counts.most_common(top_n)
ids, cnts = zip(*top)
names = [CLASSES[i] if i < len(CLASSES) else str(i) for i in ids]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(cnts)), cnts, color='steelblue')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_title(f'Top-{top_n} classes by annotation count (val)')
ax.set_ylabel('Annotations')
plt.tight_layout()
plt.show()

## 8 — Fine-tune YOLOv8s-WorldV2

`WorldTrainer` updates both the visual backbone **and** the CLIP text encoder during training.

In [ ]:
from ultralytics import YOLO
from ultralytics.models.yolo.world.train import WorldTrainer

_last_ckpt = RUNS_DIR / EXP_NAME / 'weights' / 'last.pt'
start_from = str(_last_ckpt) if _last_ckpt.exists() else BASE_MODEL
if _last_ckpt.exists():
    print(f'Resuming from checkpoint: {_last_ckpt}')
else:
    print(f'Starting from: {BASE_MODEL}  (auto-downloaded if not cached)')

model = YOLO(start_from)

results = model.train(
    trainer       = WorldTrainer,
    data          = str(DATA_YAML_PATH),
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    workers       = WORKERS,
    device        = DEVICE,
    project       = str(RUNS_DIR),
    name          = EXP_NAME,
    resume        = _last_ckpt.exists(),
    lr0           = 1e-4,
    lrf           = 0.01,
    warmup_epochs = 3,
    weight_decay  = 0.0005,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.5,
    mosaic        = 1.0,
    mixup         = 0.15,
    copy_paste    = 0.1,
    exist_ok      = True,
    verbose       = True,
    # freeze = 10,   # uncomment to freeze backbone and save GPU memory
)

BEST_CKPT = Path(results.save_dir) / 'weights' / 'best.pt'
print('\nTraining complete.')
print('Best checkpoint:', BEST_CKPT)

## 9 — Evaluate

In [ ]:
from ultralytics import YOLO

eval_model = YOLO(str(BEST_CKPT))
metrics = eval_model.val(
    data    = str(DATA_YAML_PATH),
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    device  = DEVICE,
    split   = 'val',
    verbose = False,
)

print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 10 — Visualise Predictions

In [ ]:
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

vis_model = YOLO(str(BEST_CKPT))

IMG_VAL = DATASET_ROOT / 'valid' / 'images'
if not IMG_VAL.exists():
    IMG_VAL = DATASET_ROOT / 'val' / 'images'

val_images = sorted(IMG_VAL.glob('*.jpg'))
samples = random.sample(val_images, min(6, len(val_images)))

preds = vis_model.predict(
    source  = [str(p) for p in samples],
    imgsz   = IMG_SIZE,
    conf    = 0.35,
    device  = DEVICE,
    save    = False,
    verbose = False,
)

rows, cols = 2, 3
fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
for ax, result in zip(axes.flat, preds):
    ax.imshow(result.plot()[:, :, ::-1])
    ax.axis('off')
    n = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f'{Path(result.path).name}  —  {n} det', fontsize=8)
for ax in axes.flat[len(preds):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 11 — Export

In [ ]:
from ultralytics import YOLO

export_model = YOLO(str(BEST_CKPT))

# ONNX — universal, works on CPU / ONNX Runtime / TensorRT
onnx_path = export_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    opset    = 17,
    simplify = True,
    dynamic  = False,
)
print('ONNX exported to:', onnx_path)

# Other formats (uncomment as needed):
# export_model.export(format='torchscript', imgsz=IMG_SIZE)          # embedded/mobile
# export_model.export(format='engine', imgsz=IMG_SIZE, half=True, device=0)  # TensorRT
# export_model.export(format='ncnn', imgsz=IMG_SIZE)                 # Raspberry Pi / ARM

## 12 — Download Results from Colab

Downloads `.pt` and `.onnx` to your local machine. Drop them into `backend/models/`.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import files as colab_files
    IN_COLAB_ENV = True
except ImportError:
    IN_COLAB_ENV = False

OUTPUT_NAME_PT   = 'yolov8s_world_obj365.pt'
OUTPUT_NAME_ONNX = 'yolov8s_world_obj365.onnx'

pt_out   = Path('/content') / OUTPUT_NAME_PT
onnx_out = Path('/content') / OUTPUT_NAME_ONNX

shutil.copy2(BEST_CKPT, pt_out)
print(f'PT model  : {pt_out}')

_onnx_src = BEST_CKPT.with_suffix('.onnx')
if _onnx_src.exists():
    shutil.copy2(_onnx_src, onnx_out)
    print(f'ONNX model: {onnx_out}')

if IN_COLAB_ENV:
    print('\nTriggering browser download...')
    colab_files.download(str(pt_out))
    if onnx_out.exists():
        colab_files.download(str(onnx_out))
else:
    print('\nNot in Colab — copy files from the paths above.')

print()
print(f'Place the file in backend/models/ then set:')
print(f'  _MODEL_FILENAME = "{OUTPUT_NAME_PT}"  # in backend/app.py')

---
## Appendix — Tips

### Running out of GPU memory (T4 = 15 GB)
```python
BATCH  = 8     # halve the batch
freeze = 10    # add to model.train() to freeze first 10 backbone layers
```

### Download stalls / stuck at X%
Just **re-run Section 5** — `wget -c` resumes from where it left off.  
If the URL has expired (pre-signed URLs are time-limited), re-run from the top of Section 5 to fetch a fresh URL.

### Corrupt zip error
Delete `roboflow_obj365.zip` and the `.done` sentinel, then re-run Section 5:
```python
import os
os.remove(DATASET_DIR / 'roboflow_obj365.zip')
os.remove(EXTRACT_DIR / '.done')
```

### Resume training after a Colab disconnect
Re-run all cells — the notebook auto-detects `last.pt` and resumes from where it left off.  
Mount Drive (Section 2) to keep checkpoints safe across disconnects.

### Roboflow dataset versions
Change `DATASET_VERSION` in Section 3. All versions: https://universe.roboflow.com/objects-365-consortium/objects-365

### Multi-GPU (Colab Pro+)
```python
DEVICE = '0,1'
```